# QE Tutorial — Option B (new session)

Use this cell at the top of any tutorial notebook when starting a **new Colab session** (runtime was reset, disconnected, or this is a different day).

It will:
1. Install condacolab and **restart the kernel** (expected — re-run after restart)
2. Restore `qe_env` from your Google Drive archive (~1 min)
3. Add `qe_env` site-packages to `sys.path` for Python imports
4. Install `ovito` via pip (compatible with Colab's Python 3.12)

> ⚠️ **After the kernel restarts, re-run this cell from the top.** The second run skips the install and goes straight to the restore step.

In [ ]:
# ── Step 1: Bootstrap condacolab (triggers kernel restart on first run) ────────
try:
    import condacolab
    condacolab.check()
    print("✅ condacolab active — continuing to restore …")
except Exception:
    import subprocess, sys
    print("Installing condacolab …")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "condacolab"],
        stdout=subprocess.DEVNULL
    )
    import condacolab
    condacolab.install()    # ← kernel restarts here; re-run cell after restart

In [ ]:
# ── Step 2: Restore qe_env from Google Drive ──────────────────────────────────
from google.colab import drive
import subprocess, os

drive.mount("/content/drive")

ENV_ARCHIVE = "/content/drive/MyDrive/conda_envs/qe_env.tar.gz"
ENV_PATH    = "/usr/local/envs/qe_env"

if not os.path.isdir(ENV_PATH):
    print("Restoring qe_env from Drive (~1 min) …")
    os.makedirs(ENV_PATH, exist_ok=True)
    subprocess.run(
        ["tar", "-xzf", ENV_ARCHIVE, "-C", ENV_PATH],
        check=True
    )
    print("✅ Environment restored.")
else:
    print("✅ qe_env already present on disk.")

In [ ]:
# ── Step 3: Expose Python packages and install ovito ─────────────────────────
import sys, glob, subprocess

# Add qe_env site-packages to sys.path (ase, numpy, matplotlib, …)
for sp in glob.glob("/usr/local/envs/qe_env/lib/python*/site-packages"):
    if sp not in sys.path:
        sys.path.insert(0, sp)
        print(f"✅ Added to sys.path: {sp}")

# Install ovito via pip — conda version is compiled for Python 3.11
# but Colab runs Python 3.12, so pip is the correct install method
print("Installing ovito via pip …")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "ovito"],
    stdout=subprocess.DEVNULL
)

# ── Verify ────────────────────────────────────────────────────────────────────
print("\nPackage availability:")
for pkg in ["numpy", "matplotlib", "ase", "ovito"]:
    try:
        __import__(pkg)
        print(f"  ✅  {pkg}")
    except ImportError as e:
        print(f"  ❌  {pkg} — {e}")

print("\nQE executables:")
for exe in ["pw.x", "ph.x", "pp.x", "bands.x", "dos.x"]:
    found = subprocess.run(
        ["conda", "run", "-n", "qe_env", "which", exe],
        capture_output=True, text=True
    )
    status = "✅" if found.returncode == 0 else "❌"
    print(f"  {status}  {exe:12s}  {found.stdout.strip()}")

print("\n🎉 Ready — run QE with: !conda run -n qe_env pw.x < input.in")

---
## Si SCF calculation

The cells below:
1. Download the Si PAW pseudopotential (`Si.pbe-n-kjpaw_psl.1.0.0.UPF`) from the QE website
2. Write a minimal SCF input for bulk Si (FCC, 2 atoms, `ecutwfc=30 Ry`)
3. Run `pw.x` and print the total energy

Key parameters:
| Parameter | Value | Meaning |
|-----------|-------|---------|
| `ibrav=2` | FCC | Bravais lattice |
| `celldm(1)=10.26` | bohr | Experimental lattice constant |
| `ecutwfc=30` | Ry | Wavefunction cutoff |
| `ecutrho=240` | Ry | Charge density cutoff (8× for PAW) |
| `K_POINTS 4 4 4` | — | Monkhorst-Pack k-grid |

In [ ]:
# ── Download Si pseudopotential ───────────────────────────────────────────────
import subprocess, os

PSEUDO_DIR  = '/content/pseudo'
PSEUDO_URL  = 'https://pseudopotentials.quantum-espresso.org/upf_files/Si.pbe-n-kjpaw_psl.1.0.0.UPF'
PSEUDO_FILE = f'{PSEUDO_DIR}/Si.pbe-n-kjpaw_psl.1.0.0.UPF'

os.makedirs(PSEUDO_DIR, exist_ok=True)

if not os.path.isfile(PSEUDO_FILE):
    print('Downloading Si pseudopotential …')
    subprocess.run(['wget', '-q', '-P', PSEUDO_DIR, PSEUDO_URL], check=True)
    print(f'✅ Saved to {PSEUDO_FILE}')
else:
    print(f'✅ Already present: {PSEUDO_FILE}')

In [ ]:
# ── Write Si SCF input file ───────────────────────────────────────────────────
import os

os.makedirs('/content/tmp', exist_ok=True)

scf_input = """\
&CONTROL
  calculation  = 'scf'
  prefix       = 'silicon'
  pseudo_dir   = '/content/pseudo'
  outdir       = '/content/tmp'
/
&SYSTEM
  ibrav        = 2
  celldm(1)    = 10.26
  nat          = 2
  ntyp         = 1
  ecutwfc      = 30.0
  ecutrho      = 240.0
/
&ELECTRONS
  conv_thr     = 1.0d-8
/
ATOMIC_SPECIES
  Si  28.086  Si.pbe-n-kjpaw_psl.1.0.0.UPF
ATOMIC_POSITIONS {alat}
  Si  0.00  0.00  0.00
  Si  0.25  0.25  0.25
K_POINTS {automatic}
  4 4 4 1 1 1
"""

with open('scf.in', 'w') as f:
    f.write(scf_input)

print('✅ scf.in written')
print(scf_input)

In [ ]:
# ── Run Si SCF calculation ────────────────────────────────────────────────────
import subprocess

print('Running pw.x SCF for Si …\n')

result = subprocess.run(
    ['conda', 'run', '-n', 'qe_env', 'pw.x', '-input', 'scf.in'],
    capture_output=True, text=True
)

with open('scf.out', 'w') as f:
    f.write(result.stdout)

# Print last 30 lines (convergence summary)
lines = result.stdout.strip().splitlines()
print('\n'.join(lines[-30:]))

if '!    total energy' in result.stdout:
    for line in lines:
        if '!    total energy' in line:
            print(f'\n🎉 {line.strip()}')
else:
    print('\n⚠️  SCF may not have converged — check scf.out')
    print(result.stderr[-500:] if result.stderr else '')